# Examen de Recuperación de Información — NFCorpus

**Estudiante:** Jhonn Saeteros  
**Asignatura:** ICCD753 — Recuperación de Información  
**Institución:** Escuela Politécnica Nacional  
**Docente:** Ing. Iván Carrera

## Enunciado

# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

## 1. Objetivo

Implementar y evaluar TF-IDF con similitud coseno y BM25 sobre NFCorpus, y ofrecer
la búsqueda mediante una API REST.

La representación de cada documento reúne su título y contenido. El título aporta
el tema y el texto incorpora los detalles. Se conserva el documento completo y su
identificador original para mantener la correspondencia con los qrels.

Fuentes: [NFCorpus](https://huggingface.co/datasets/BeIR/nfcorpus) y
[juicios de relevancia](https://huggingface.co/datasets/BeIR/nfcorpus-qrels).

## 2. Preparación

Instalar las dependencias con `python -m pip install -r requirements-dev.txt` y
seleccionar ese entorno como kernel. Los módulos utilizados son `preprocessing.py`,
`retrieval.py`, `evaluation.py` y `app.py`.

`SERVICE_URL` contiene la dirección pública del servicio cuando esté desplegado.

In [1]:
from pathlib import Path
import sys, math, subprocess
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'retrieval.py').exists())
sys.path.insert(0, str(ROOT))
from preprocessing import normalize, tokenize, preprocess, STOPWORDS
from retrieval import SearchIndex
from evaluation import load_dataset, evaluate, metrics_at_k
from tests.test_system import verify, verify_local

ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)
SERVICE_URL = ''  # URL pública del servicio en Render
print('Python:', sys.version.split()[0])

Python: 3.13.14


C:\Users\Jhonn22\OneDrive - Escuela Politécnica Nacional\Escritorio\!Jhonn Trabajos!\RECUPERACION DE LA INFORMACION\examen1bim\examen1bimJhonnSaeteros\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 3. Carga y exploración de NFCorpus

Los archivos locales están en `nfcorpus/nfcorpus`. Se cargan todos los documentos
y consultas, y los juicios de la partición `test`. La carga verifica los tipos,
los identificadores únicos y las referencias de los juicios. Los identificadores
se mantienen como cadenas de texto.

In [2]:
documents, queries, qrels = load_dataset()
assert len(documents) == 3633 and len(queries) == 3237
assert len(qrels) == 323 and sum(map(len, qrels.values())) == 12334
display(pd.DataFrame([{
    'Documentos': len(documents), 'Consultas totales': len(queries),
    'Consultas test': len(qrels), 'Juicios test': sum(map(len, qrels.values())),
    'Grados de relevancia': sorted({g for judgments in qrels.values() for g in judgments.values()})}]))
print('Campos de documento:', list(documents[0]))
display(pd.DataFrame(documents[:2])[['_id','title','text']])
display(pd.DataFrame([{'query_id': qid, 'query': queries[qid],
    'judgments': len(qrels[qid])} for qid in sorted(qrels)[:4]]))
first_qid = 'PLAIN-3332'
print('Qrels de ejemplo:', first_qid, list(qrels[first_qid].items())[:5])

,Documentos,Consultas totales,Consultas test,Juicios test,Grados de relevancia
0,3633,3237,323,12334,"[1, 2]"


Campos de documento: ['_id', 'title', 'text', 'metadata']


,_id,title,text
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an..."
1,MED-14,Statin use after diagnosis of breast cancer an...,BACKGROUND: Preclinical studies have shown tha...


,query_id,query,judgments
0,PLAIN-1008,deafness,5
1,PLAIN-1018,DHA,60
2,PLAIN-102,Stopping Heart Disease in Childhood,24
3,PLAIN-1028,dietary scoring,1


Qrels de ejemplo: PLAIN-3332 [('MED-4994', 2), ('MED-3696', 1), ('MED-3697', 1), ('MED-3698', 1), ('MED-3699', 1)]


## 4. Preprocesamiento

Se aplica el mismo procesamiento a documentos y consultas:

1. Normalización Unicode NFKC y conversión a minúsculas.
2. Tokenización conservando decimales, números con signo, letras griegas,
   términos como `B12` y símbolos como `%`, `>=` y `±`.
3. Eliminación de stopwords inglesas. Se conservan `no`, `nor`, `not`, `without`
   y `never`, por su relación con la negación. Las contracciones `n't` se expanden a `not`.
4. Stemming Snowball para agrupar variantes morfológicas sin diccionarios externos.

La puntuación actúa como separador, salvo los símbolos incluidos en la regla.
Los compuestos con guion se separan; un guion seguido de un número se conserva
como signo. El stemming puede unir palabras con sentidos distintos.
El texto original se conserva para presentar los resultados.

In [3]:
examples = [documents[0]['title'] + ' ' + documents[0]['text'][:230],
    "Vitamin B12: 2.5 mg; NOT without β-carotene, risk >= 5%. Isn't harmful.",
    queries[first_qid]]
display(pd.DataFrame([{'Original': text, 'Unicode/minúsculas': normalize(text),
    'Tokens': tokenize(text), 'Sin stopwords': [t for t in tokenize(text) if t not in STOPWORDS],
    'Stemming': preprocess(text)} for text in examples]))

,Original,Unicode/minúsculas,Tokens,Sin stopwords,Stemming
0,Statin Use and Breast Cancer Survival: A Natio...,statin use and breast cancer survival: a natio...,"[statin, use, and, breast, cancer, survival, a...","[statin, use, breast, cancer, survival, nation...","[statin, use, breast, cancer, surviv, nationwi..."
1,"Vitamin B12: 2.5 mg; NOT without β-carotene, r...","vitamin b12: 2.5 mg; not without β-carotene, r...","[vitamin, b12, 2.5, mg, not, without, β, carot...","[vitamin, b12, 2.5, mg, not, without, β, carot...","[vitamin, b12, 2.5, mg, not, without, β, carot..."
2,Alcohol Risks vs. Benefits,alcohol risks vs. benefits,"[alcohol, risks, vs, benefits]","[alcohol, risks, vs, benefits]","[alcohol, risk, vs, benefit]"


## 5. Vocabulario e índice invertido

`vocabulary` asigna una columna a cada término. `postings[t]` almacena pares
`(posición del documento, frecuencia)`. `doc_ids` relaciona cada posición con el
identificador original. Las frecuencias se guardan en una matriz dispersa.

- **N:** número de documentos.
- **TF:** frecuencia del término en el documento.
- **DF:** número de documentos que contienen el término.
- **Longitud:** número de tokens después del procesamiento.
- **avgdl:** longitud media de los documentos.

La unión de postings de los términos de la consulta determina los candidatos.
Los resultados se ordenan por puntuación descendente y se desempatan por ID.
El índice se guarda en `artifacts/index/index.json` y `artifacts/index/tf.npz`.

In [4]:
index = SearchIndex(documents)
index.save(ARTIFACTS / 'index')
display(pd.DataFrame([{'N': index.N, 'Vocabulario': len(index.vocabulary),
    'Pares término/documento': index.tf.nnz, 'avgdl': index.avgdl,
    'Longitud mínima': index.doc_lengths.min(), 'Longitud máxima': index.doc_lengths.max()}]))
display(pd.DataFrame([{'term': term, 'column': col, 'DF': int(index.df[col])}
    for term, col in list(index.vocabulary.items())[:12]]))
sample_term = preprocess('cholesterol')[0]
display(pd.DataFrame([{'term': sample_term, 'position': int(pos),
    'document_id': index.doc_ids[pos], 'TF': int(tf), 'length': int(index.doc_lengths[pos])}
    for pos, tf in index.postings[sample_term][:8]]))
assert index.doc_ids == [d['_id'] for d in documents]
assert np.all(index.doc_lengths == index.tf.sum(axis=1).A1)

,N,Vocabulario,Pares término/documento,avgdl,Longitud mínima,Longitud máxima
0,3633,22797,365019,166.370493,11,1063


,term,column,DF
0,%,0,1580
1,+,1,312
2,-0,2,1
3,-0.0,3,1
4,-0.0.98,4,1
5,-0.00,5,2
6,-0.002,6,1
7,-0.003,7,1
8,-0.00384,8,1
9,-0.004,9,1


,term,position,document_id,TF,length
0,cholesterol,20,MED-711,4,228
1,cholesterol,21,MED-712,1,184
2,cholesterol,33,MED-726,3,177
3,cholesterol,50,MED-751,7,208
4,cholesterol,53,MED-754,4,351
5,cholesterol,61,MED-816,2,168
6,cholesterol,67,MED-823,1,201
7,cholesterol,79,MED-835,1,207


## 6. TF-IDF con similitud coseno

Para un término con frecuencia positiva:

$$TF(t,d)=1+\ln f(t,d),\qquad IDF(t)=1+\ln\frac{N+1}{df(t)+1}$$

$$w_{t,d}=TF(t,d)\,IDF(t),\qquad
\cos(q,d)=\frac{\sum_t w_{t,q}w_{t,d}}{\|q\|_2\|d\|_2}.$$

Los términos ausentes tienen peso cero. La consulta utiliza el vocabulario y el
IDF del corpus; los términos desconocidos se ignoran. Las normas de los documentos
se precalculan y las divisiones con denominador cero se evitan.

**Ejemplo de comprobación:** D1 = `apple apple banana`, D2 = `banana carrot`,
q = `apple banana`. N = 2, df(apple) = 1 y df(banana) = 2. Por tanto,
IDF(apple) = 1 + ln(3/2) e IDF(banana) = 1. Se compara el cálculo manual
del coseno de D1 con el resultado del modelo.

In [5]:
tiny = SearchIndex([{'_id':'D1','title':'','text':'apple apple banana'},
                    {'_id':'D2','title':'','text':'banana carrot'}])
ia = 1 + math.log(3/2)
wa = (1 + math.log(2)) * ia
manual_cosine = (wa * ia + 1) / (math.sqrt(wa**2 + 1) * math.sqrt(ia**2 + 1))
actual_cosine = tiny.search('apple banana', 'tfidf', 2)
assert math.isclose(actual_cosine[0]['score'], manual_cosine, abs_tol=1e-12)
print('Coseno D1 manual:', manual_cosine)
display(pd.DataFrame(actual_cosine))
print('Vector disperso q:', tiny.query_vector('apple banana').toarray())

Coseno D1 manual: 0.9757694105051148


,rank,document_id,score,title,snippet
0,1,D1,0.975769,,apple apple banana
1,2,D2,0.336097,,banana carrot


Vector disperso q: [[1.40546511 1.         0.        ]]


## 7. BM25

$$IDF(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)$$

$$BM25(q,d)=\sum_{t\in unique(q)}IDF(t)
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/avgdl)}.$$

Se utilizan **k1 = 1.2** y **b = 0.75**, sin ajuste con los qrels de prueba.
k1 controla la saturación de la frecuencia: las repeticiones aportan incrementos
cada vez menores. b regula la normalización según la longitud del documento.
IDF da mayor peso a los términos menos frecuentes en el corpus.

BM25 normaliza la frecuencia por longitud, mientras el coseno normaliza el vector
completo. Cada término distinto de la consulta aporta una contribución a BM25.

En el ejemplo anterior, avgdl = 2.5, IDF(apple) = ln(2) e IDF(banana) = ln(1.2).
Se comprueba la puntuación de D1 mediante el cálculo de sus dos contribuciones.

In [6]:
manual_bm25 = math.log(2)*2*2.2/(2+1.2*(.25+.75*3/2.5))
manual_bm25 += math.log(1.2)*2.2/(1+1.2*(.25+.75*3/2.5))
actual_bm25 = tiny.search('apple banana','bm25',2)
assert math.isclose(actual_bm25[0]['score'],manual_bm25,abs_tol=1e-12)
print('BM25 D1 manual:', manual_bm25)
display(pd.DataFrame(actual_bm25))
display(pd.DataFrame(tiny.explain('apple banana','D1')))

BM25 D1 manual: 1.0708543050001982


,rank,document_id,score,title,snippet
0,1,D1,1.070854,,apple apple banana
1,2,D2,0.198568,,banana carrot


,term,tf,df,length,tfidf_contribution,bm25_contribution
0,appl,2,1,3,0.751172,0.902322
1,banana,1,2,3,0.224597,0.168533


## 8. Recuperación de documentos

Se comparan los primeros cinco resultados de una consulta oficial y una consulta
libre. Las puntuaciones de los dos modelos tienen escalas distintas; la comparación
se realiza por el orden y la relevancia de los documentos recuperados.
Si no existen coincidencias léxicas, el resultado es una lista vacía.

In [7]:
for text in [queries[first_qid], 'fermented chickpeas exercise inflammation 2026']:
    display(Markdown(f'**Consulta:** {text}'))
    for model in ['tfidf','bm25']:
        print(model)
        display(pd.DataFrame(index.search(text,model,5)))
print('Fuera de vocabulario:', index.search('zxqvprsnm','bm25',10))

**Consulta:** Alcohol Risks vs. Benefits

tfidf


,rank,document_id,score,title,snippet
0,1,MED-4994,0.191145,Who benefits most from the cardioprotective pr...,BACKGROUND: The cardioprotective properties of...
1,2,MED-3696,0.188357,Alcohol consumption and breast cancer risk in ...,The authors assessed the association between m...
2,3,MED-3762,0.186291,Moderate alcohol consumption during adult life...,Context Multiple studies have linked alcohol c...
3,4,MED-3768,0.179868,The positive and negative health effects of al...,"In this paper, the negative and the positive e..."
4,5,MED-3766,0.174148,Epidemiology and pathophysiology of alcohol an...,AIMS: To update epidemiological data on alcoho...


bm25


,rank,document_id,score,title,snippet
0,1,MED-1680,11.241494,Effect of potentially modifiable risk factors ...,BACKGROUND: Although more than 80% of the glob...
1,2,MED-3696,10.284088,Alcohol consumption and breast cancer risk in ...,The authors assessed the association between m...
2,3,MED-4994,9.837812,Who benefits most from the cardioprotective pr...,BACKGROUND: The cardioprotective properties of...
3,4,MED-3763,9.440966,A single sip of a strong alcoholic beverage ca...,The aim of this study was to explore oral expo...
4,5,MED-1678,9.201869,Combined effect of low-risk dietary and lifest...,BACKGROUND: Limited data are available on the ...


**Consulta:** fermented chickpeas exercise inflammation 2026

tfidf


,rank,document_id,score,title,snippet
0,1,MED-2009,0.242176,Nutritional quality and health benefits of chi...,Chickpea (Cicer arietinum L.) is an important ...
1,2,MED-5204,0.168295,Relevance of protein fermentation to gut health.,It is generally accepted that carbohydrate fer...
2,3,MED-4125,0.152376,Human gut microbiota does not ferment erythritol.,"Erythritol, a naturally occurring polyol, is g..."
3,4,MED-3583,0.141238,First and second meal effects of pulses on blo...,Pulses are low-glycemic appetite-suppressing f...
4,5,MED-2298,0.137344,A guide to exercise prescription.,Exercise is a fundamental component of good he...


bm25


,rank,document_id,score,title,snippet
0,1,MED-2009,11.994529,Nutritional quality and health benefits of chi...,Chickpea (Cicer arietinum L.) is an important ...
1,2,MED-4269,9.986521,Butyrate and other short-chain fatty acids as ...,PURPOSE OF REVIEW: High-fiber diets have been ...
2,3,MED-3583,9.945481,First and second meal effects of pulses on blo...,Pulses are low-glycemic appetite-suppressing f...
3,4,MED-3450,9.338472,"Exercise-induced oxidative stress:myths, reali...",Although assays for the most popular markers o...
4,5,MED-3465,9.026385,Effect of blueberry ingestion on natural kille...,Blueberries are rich in antioxidants known as ...


Fuera de vocabulario: []


## 9. Evaluación

Se evalúan las **323 consultas de `qrels/test.tsv`**. Para Precision y Recall,
un documento es relevante si su grado es mayor que cero. Los documentos sin juicio
reciben grado cero en el cálculo.

$$P@10=\frac{\#\text{ relevantes en top 10}}{10},\qquad
R@10=\frac{\#\text{ relevantes en top 10}}{\#\text{ relevantes de la consulta}}$$

$$DCG@10=\sum_{i=1}^{10}\frac{2^{rel_i}-1}{\log_2(i+1)},\qquad
NDCG@10=\frac{DCG@10}{IDCG@10}.$$

NDCG conserva los grados 1 y 2 con ganancia $2^{rel}-1$. IDCG se obtiene ordenando
todos los juicios de la consulta por relevancia. Las posiciones no recuperadas
aportan cero; Precision siempre divide para diez. Si no hay relevantes, Recall
es cero; si IDCG es cero, NDCG también es cero.

El promedio es macro: cada consulta tiene el mismo peso. Los qrels se consultan
después de obtener el ranking. Los resultados por consulta se guardan en
`artifacts/metrics_per_query.csv` y los promedios en `artifacts/metrics_summary.csv`.

In [8]:
hand = metrics_at_k(['B','X','A'], {'A':2,'B':1,'C':1},3)
expected_ndcg = (1+3/math.log2(4))/(3+1/math.log2(3)+1/math.log2(4))
assert math.isclose(hand['NDCG@3'],expected_ndcg)
assert hand['Precision@3'] == 2/3 and hand['Recall@3'] == 2/3
assert metrics_at_k([], {}) == {'Precision@10':0.,'Recall@10':0.,'NDCG@10':0.}
print('Ejemplo manual de métricas:', hand)
per_query, summary, runs = evaluate(index, queries, qrels, 10)
per_query.to_csv(ARTIFACTS / 'metrics_per_query.csv', index=False)
summary.to_csv(ARTIFACTS / 'metrics_summary.csv')
display(summary.rename_axis('Modelo').rename(index={'tfidf':'TF-IDF','bm25':'BM25'}).style.format('{:.6f}'))
display(per_query.head(10))
print('Consultas por modelo:', per_query.groupby('model').size().to_dict())

Ejemplo manual de métricas:

 {'Precision@3': 0.6666666666666666, 'Recall@3': 0.6666666666666666, 'NDCG@3': 0.6051906348295047}


,Precision@10,Recall@10,NDCG@10
Modelo,,,
TF-IDF,0.232508,0.153092,0.323104
BM25,0.238080,0.152893,0.328409


,model,query_id,Precision@10,Recall@10,NDCG@10
0,tfidf,PLAIN-1008,0.0,0.000000,0.000000
1,tfidf,PLAIN-1018,0.6,0.100000,0.636474
2,tfidf,PLAIN-102,0.0,0.000000,0.000000
3,tfidf,PLAIN-1028,0.0,0.000000,0.000000
4,tfidf,PLAIN-1039,0.2,1.000000,1.000000
5,tfidf,PLAIN-1050,0.1,0.010638,0.063621
6,tfidf,PLAIN-1066,0.0,0.000000,0.000000
7,tfidf,PLAIN-1088,0.2,0.400000,0.553146
8,tfidf,PLAIN-1098,0.0,0.000000,0.000000
9,tfidf,PLAIN-1109,0.5,0.051020,0.620397


Consultas por modelo: {'bm25': 323, 'tfidf': 323}


## 10. Análisis de resultados

Los siguientes casos muestran las mayores diferencias de NDCG@10 y un caso de
bajo rendimiento de ambos modelos. Se incluyen el grado de relevancia, la longitud
del documento y las contribuciones de los términos.

**TF-IDF — `PLAIN-3332`, Alcohol Risks vs. Benefits.** TF-IDF coloca `MED-4994`
(grado 2) primero y BM25 lo coloca tercero. BM25 sitúa primero `MED-1680`, sin
juicio positivo para esta consulta: las frecuencias de `alcohol`, `risk` y `vs`
producen una suma elevada, pese a su mayor longitud.

**BM25 — `PLAIN-3342`, Is Coconut Milk Good For You?** BM25 sitúa primero
`MED-5012` y segundo `MED-5014`, ambos con grado 2. TF-IDF coloca primero
`MED-5015`, un documento corto sobre el uso intravenoso de agua de coco, sin juicio
positivo en esta consulta. La coincidencia de `coconut` y la norma del documento
favorecen su coseno; BM25 aprovecha otras coincidencias en los documentos relevantes.

**Limitación compartida — `PLAIN-1008`, deafness.** Ambos recuperan únicamente
`MED-4668`, sin juicio positivo. La raíz `deaf` aparece en un documento y no
coincide con los cinco documentos relevantes. El procesamiento morfológico no
resuelve por sí solo los sinónimos ni las relaciones entre conceptos.
Un documento sin juicio se cuenta con grado cero, aunque eso no permite descartar
que tenga relación con el tema.

In [9]:
comparison = per_query.pivot(index='query_id',columns='model',values='NDCG@10')
comparison['delta_tfidf_bm25'] = comparison['tfidf'] - comparison['bm25']
cases = {'TF-IDF superior': comparison['delta_tfidf_bm25'].idxmax(),
         'BM25 superior': comparison['delta_tfidf_bm25'].idxmin(),
         'Ambos limitados': comparison[['tfidf','bm25']].max(axis=1).idxmin()}
display(comparison.loc[list(cases.values())])
for label, qid in cases.items():
    display(Markdown(f'### {label}: {qid} — {queries[qid]}'))
    for model in ['tfidf','bm25']:
        enriched = [{**r, 'grade': qrels[qid].get(r['document_id'],0),
            'length': int(index.doc_lengths[index.id_to_position[r['document_id']]])}
            for r in runs[model][qid]]
        print(model)
        display(pd.DataFrame(enriched)[['rank','document_id','score','grade','length','title']])
        for r in runs[model][qid][:1]:
            print('Contribuciones:', r['document_id'])
            display(pd.DataFrame(index.explain(queries[qid],r['document_id'])))
        if len(runs[model][qid]) < 10:
            print('Menos de 10 candidatos; las posiciones restantes aportan cero.')

model,bm25,tfidf,delta_tfidf_bm25
query_id,,,
PLAIN-3332,0.439680,0.848619,0.408939
PLAIN-3342,0.765361,0.498189,-0.267171
PLAIN-1008,0.000000,0.000000,0.000000


### TF-IDF superior: PLAIN-3332 — Alcohol Risks vs. Benefits

tfidf


,rank,document_id,score,grade,length,title
0,1,MED-4994,0.191145,2,137,Who benefits most from the cardioprotective pr...
1,2,MED-3696,0.188357,1,175,Alcohol consumption and breast cancer risk in ...
2,3,MED-3762,0.186291,1,180,Moderate alcohol consumption during adult life...
3,4,MED-3768,0.179868,1,97,The positive and negative health effects of al...
4,5,MED-3766,0.174148,1,172,Epidemiology and pathophysiology of alcohol an...
5,6,MED-1680,0.170462,0,273,Effect of potentially modifiable risk factors ...
6,7,MED-4445,0.169096,0,178,Alcohol intake and mortality among women with ...
7,8,MED-3765,0.162777,1,63,Molecular mechanisms of alcohol-mediated carci...
8,9,MED-1678,0.160941,0,200,Combined effect of low-risk dietary and lifest...
9,10,MED-3763,0.150390,1,158,A single sip of a strong alcoholic beverage ca...


Contribuciones: MED-4994


,term,tf,df,length,tfidf_contribution,bm25_contribution
0,alcohol,4,166,137,0.107396,5.381987
1,benefit,5,314,137,0.083749,4.455824


... (19 outputs left out)

,rank,document_id,score,grade,length,title
0,1,MED-4668,9.046572,0,110,Cretinism revisited.


Contribuciones: MED-4668


,term,tf,df,length,tfidf_contribution,bm25_contribution
0,deaf,1,1,110,0.151697,9.046572


Menos de 10 candidatos; las posiciones restantes aportan cero.


## 11. Servicio web

La API importa `SearchIndex` desde `retrieval.py` y carga el índice al iniciar.
Los qrels se utilizan únicamente en la evaluación.

- `GET /health`: HTTP 200 cuando el índice está cargado.
- `POST /search`: recibe `query`, `model` (`tfidf` o `bm25`) y `top_k`.
- La respuesta contiene `model` y `results`; cada resultado incluye `rank`,
  `document_id`, `score`, `title` y `snippet`.
- Entradas inválidas: HTTP 422. Consulta sin coincidencias: HTTP 200 y lista vacía.

Las pruebas comprueban las fórmulas, el guardado y carga del índice, los IDs,
el orden de los resultados, top_k y la validación de las solicitudes.

In [10]:
tests = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider'],
    cwd=ROOT, capture_output=True, text=True)
if tests.returncode:
    print(tests.stdout, tests.stderr)
assert tests.returncode == 0
print(tests.stdout.strip().splitlines()[-1])

30 passed, 1 warning in 7.80s


## 12. Pruebas HTTP

Se inicia un servidor local temporal y se envían 23 solicitudes HTTP. Se prueban
ambos modelos, consultas oficiales y nuevas, diferentes valores de top_k y entradas
inválidas. Los resultados de la API se comparan con los del notebook.
El servidor de prueba se cierra al finalizar.

In [11]:
http_report = verify_local()
print('Comprobaciones HTTP correctas:', http_report['checks'])
for model in ['tfidf', 'bm25']:
    example = next(r for r in http_report['records'] if
        r.get('request', {}).get('model') == model and
        r['request']['top_k'] == 10 and r['status'] == 200)
    print(model, 'HTTP', example['status'], 'Consulta:', example['request']['query'])
    display(pd.DataFrame(example['body']['results']))

Comprobaciones HTTP correctas: 23
tfidf HTTP 200 Consulta: deafness


,rank,document_id,score,title,snippet
0,1,MED-4668,0.151697,Cretinism revisited.,Endemic cretinism includes two syndromes: a mo...


bm25 HTTP 200 Consulta: deafness


,rank,document_id,score,title,snippet
0,1,MED-4668,9.046572,Cretinism revisited.,Endemic cretinism includes two syndromes: a mo...


## 13. Despliegue en Render

La configuración del servicio está en `render.yaml`. El índice se construye durante
el build y se carga al iniciar la API. El servidor escucha en `0.0.0.0` y en el
puerto definido por `PORT`.

La dirección pública se registra en `SERVICE_URL`, en la sección 2. La siguiente
celda comprueba `/health` y `/search` con ambos modelos.
El plan gratuito puede requerir una espera de arranque tras un período de inactividad.
[Documentación de Render](https://render.com/docs/free).

In [12]:
if SERVICE_URL:
    public_report = verify(SERVICE_URL)
    print('URL pública:', SERVICE_URL)
    print('Verificación UTC:', public_report['verified_at_utc'])
    print('Comprobaciones correctas:', public_report['checks'])
else:
    print('Estado del despliegue: pendiente. SERVICE_URL sin configurar.')

Estado del despliegue: pendiente. SERVICE_URL sin configurar.


## 14. Conclusiones

La comparación global se realiza con los promedios de las 323 consultas de prueba.
Los casos individuales muestran que la normalización y el tratamiento de la
frecuencia pueden cambiar las primeras posiciones del ranking. La falta de
coincidencia de vocabulario limita a ambos modelos.

In [13]:
display(summary.rename(index={'tfidf': 'TF-IDF', 'bm25': 'BM25'}).style.format('{:.6f}'))
tf, bm = summary.loc['tfidf'], summary.loc['bm25']
delta = bm['NDCG@10'] - tf['NDCG@10']
display(Markdown(
    f"BM25 obtuvo Precision@10 de **{bm['Precision@10']:.6f}** y NDCG@10 de "
    f"**{bm['NDCG@10']:.6f}**, frente a **{tf['Precision@10']:.6f}** y "
    f"**{tf['NDCG@10']:.6f}** de TF-IDF. La diferencia de NDCG@10 fue "
    f"**{delta:.6f}** a favor de BM25. TF-IDF obtuvo un Recall@10 ligeramente "
    f"mayor: **{tf['Recall@10']:.6f}**, frente a **{bm['Recall@10']:.6f}**."
))
assert len(per_query) == 646 and tests.returncode == 0 and http_report['checks'] == 23

,Precision@10,Recall@10,NDCG@10
model,,,
TF-IDF,0.232508,0.153092,0.323104
BM25,0.238080,0.152893,0.328409


BM25 obtuvo Precision@10 de **0.238080** y NDCG@10 de **0.328409**, frente a **0.232508** y **0.323104** de TF-IDF. La diferencia de NDCG@10 fue **0.005304** a favor de BM25. TF-IDF obtuvo un Recall@10 ligeramente mayor: **0.153092**, frente a **0.152893**.